<a href="https://colab.research.google.com/github/bellDataSc/hacktoberfest-FUSP-2026/blob/main/Laya_vs_Gemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laya × Gemma: quando a IA pode decidir e quando precisa de revisão humana?

Coloco dois modelos abertos para fazer a mesma triagem de chamados em português: o **Laya**, que devolve uma distribuição de probabilidade sem gerar texto, e o **Gemma 4**, que responde em JSON. Uso a partição de ajuste para calibrar o Laya e escolher o limiar de revisão, e meço tudo na partição de avaliação.

Eu deixo os 180 chamados dentro deste notebook. Na primeira execução, rodo os dois modelos numa GPU do Colab e salvo tudo em `resultados/`. Nas próximas, leio os resultados salvos.

*bel - para o hackathon com a FLUSP*

In [ ]:
import json, platform, subprocess, sys, time, unicodedata
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

REFAZER = False
ERRO_ALVO = 0.10
LAYA_VERSAO = "0.4.19"
LAYA_CHECKPOINT = "multilingual"
GEMMA_MODELO = "google/gemma-4-E2B-it"
GITHUB = "https://raw.githubusercontent.com/bellDataSc/hacktoberfest-FUSP-2026/main/"
PASTA = Path("resultados")
PASTA.mkdir(exist_ok=True)

CATEGORIAS = {
    "cobranca": "faturas, boletos, pagamentos, estornos, reembolsos e taxas",
    "acesso": "login, senha, código de verificação, conta bloqueada e permissões de usuários",
    "tecnico": "erros, falhas, lentidão e travamentos do aplicativo, do site ou de integrações",
    "entrega": "pedidos, rastreio, atrasos, produtos danificados ou errados, trocas e endereço de entrega",
    "cancelamento": "pedido explícito para cancelar ou encerrar assinatura, plano, conta, contrato ou pedido",
    "sem_informacao": "o texto não traz informação suficiente para saber qual equipe deve atender",
}
ROTULOS = list(CATEGORIAS)
INSTRUCAO = "Qual equipe deve tratar este chamado de atendimento ao cliente?"
REVISAR_SEMPRE = ["sem_informacao"]

AZUL, LARANJA, VERMELHO, CINZA = "#2a78d6", "#eb6834", "#d03b3b", "#c3c2b7"
TINTA, TINTA_SUAVE, GRADE, FUNDO = "#0b0b0b", "#52514e", "#e6e6e3", "#fcfcfb"
plt.rcParams.update({"figure.facecolor": FUNDO, "axes.facecolor": FUNDO, "axes.edgecolor": GRADE,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": GRADE, "axes.axisbelow": True, "xtick.color": TINTA_SUAVE,
                     "ytick.color": TINTA_SUAVE, "axes.labelcolor": TINTA_SUAVE, "axes.titlelocation": "left",
                     "axes.titlesize": 11, "legend.frameon": False, "legend.fontsize": 9})
pd.set_option("display.max_colwidth", 110)


def ler_csv(nome):
    local = Path(nome)
    return pd.read_csv(local if local.exists() else GITHUB + nome)


def ler_json(nome):
    local = Path(nome)
    return json.loads(local.read_text(encoding="utf-8") if local.exists() else urlopen(GITHUB + nome).read())


def pct(x):
    return f"{x:.1%}".replace(".", ",")


def salvar(fig, nome):
    fig.savefig(PASTA / nome, dpi=160, bbox_inches="tight")

## Dados

Escrevi 180 chamados com negação, ambiguidade e falta de informação. Separo 72 para ajuste e 108 para avaliação, com a mesma semente sempre.

In [ ]:
CHAMADOS = {
    ("cobranca", "direto"): [
        "Fui cobrada duas vezes na fatura de setembro, preciso do estorno.",
        "O boleto deste mês veio com valor maior do que o contratado.",
        "Quero a segunda via da fatura de outubro.",
        "Paguei o Pix ontem e o sistema ainda mostra a conta em aberto.",
        "Apareceu uma taxa de R$ 19,90 que eu não reconheço no meu cartão.",
        "Como faço para mudar a data de vencimento da minha fatura?",
        "Preciso da nota fiscal do pagamento feito em agosto.",
        "O desconto prometido na renovação não foi aplicado na cobrança.",
        "Quero trocar o cartão de crédito cadastrado para pagamento.",
        "Solicito reembolso da mensalidade cobrada após o período de teste.",
        "A fatura veio em nome da empresa errada, preciso corrigir o CNPJ.",
        "Quero parcelar o valor em atraso, quais são as opções?",
    ],
    ("cobranca", "negacao"): [
        "Não quero cancelar o plano, só preciso que estornem a cobrança duplicada.",
        "Não é problema de acesso, eu consigo entrar; o problema é que fui cobrado duas vezes.",
        "O aplicativo não está com erro nenhum, mas a fatura veio com um valor que não reconheço.",
        "Não recebi o reembolso que vocês prometeram há quinze dias.",
        "Meu pedido chegou certinho, não tenho reclamação da entrega; quero apenas a nota fiscal da compra.",
        "Não autorizei essa renovação automática e quero o dinheiro de volta.",
        "Ainda não caiu o estorno do pagamento duplicado.",
        "Não consigo entender por que o valor da mensalidade subiu sem aviso.",
        "Não preciso de suporte técnico, preciso de um boleto atualizado.",
        "Não encerrem minha conta: só quero trocar a forma de pagamento para boleto.",
    ],
    ("cobranca", "ambiguo"): [
        "Fui cobrado depois de pedir o cancelamento, quero o valor de volta.",
        "O pagamento foi debitado, mas o app mostrou erro e o pedido não foi confirmado.",
        "Paguei pelo frete expresso e o pedido chegou no prazo normal; quero a diferença.",
        "Minha conta foi bloqueada por falta de pagamento, mas a fatura está paga.",
        "Cobraram a assinatura de um serviço que eu já tinha encerrado.",
        "O produto veio com defeito e quero o reembolso, não a troca.",
        "Se não estornarem a cobrança duplicada até sexta, vou cancelar.",
        "Recebi um e-mail de cobrança de uma conta que não é minha.",
    ],
    ("acesso", "direto"): [
        "Esqueci minha senha e o e-mail de recuperação não chega.",
        "Minha conta foi bloqueada após três tentativas de login.",
        "O código de verificação por SMS não chega no meu celular.",
        "Troquei de número e agora não consigo receber o código de dois fatores.",
        "Quero alterar o e-mail de login da minha conta.",
        'Aparece "usuário ou senha inválidos", mas tenho certeza de que a senha está certa.',
        "Acho que alguém entrou na minha conta, recebi alerta de acesso de outro estado.",
        "Como adiciono outro usuário com acesso à conta da empresa?",
        "O link para redefinir a senha diz que expirou.",
        "Não lembro qual e-mail usei para criar a conta.",
        "Preciso remover o acesso de um funcionário que saiu da empresa.",
        "O login com Google parou de funcionar para minha conta.",
    ],
    ("acesso", "negacao"): [
        "Não quero cancelar minha conta, só não consigo entrar nela desde ontem.",
        "Não fui cobrada errado, o problema é que perdi o acesso à conta.",
        "O app não está travando; ele abre normalmente, mas não aceita minha senha.",
        "Não recebo o código de verificação de jeito nenhum.",
        "Ninguém da minha equipe consegue fazer login com as credenciais novas.",
        "Não é a entrega que me preocupa: alguém trocou o e-mail da minha conta sem eu pedir.",
        "Não reconheço esse login feito às 3h da manhã na minha conta.",
        "Não consigo redefinir a senha porque não tenho mais acesso ao e-mail antigo.",
        "Não tenho o celular cadastrado comigo, como entro sem o segundo fator?",
        "Não precisa estornar nada; só desbloqueiem meu usuário, por favor.",
    ],
    ("acesso", "ambiguo"): [
        "Depois da atualização, o app pede login toda hora e não salva a sessão.",
        "Quero encerrar o acesso do meu ex-sócio, mas manter a conta ativa.",
        "Minha conta foi suspensa e não sei se é por pagamento ou por segurança.",
        "Não consigo entrar para acompanhar meu pedido.",
        "Ao tentar entrar, dá erro de autenticação e volta para a tela inicial.",
        "Recebi um e-mail pedindo para confirmar minha senha; é golpe?",
        "Quero saber quem tem acesso às faturas da empresa.",
        "Perdi o acesso e por isso não consegui pagar a fatura em dia.",
    ],
    ("tecnico", "direto"): [
        "O aplicativo fecha sozinho toda vez que abro a tela de configurações.",
        "O site retorna erro 500 quando tento gerar o relatório.",
        "A exportação para Excel trava em 90% e não termina.",
        "As notificações pararam de chegar depois da última atualização.",
        "O sistema está muito lento desde hoje de manhã.",
        "A integração com a API devolve timeout em todas as chamadas.",
        "Os gráficos do painel aparecem em branco no Chrome.",
        'O upload de arquivos PDF falha com a mensagem "formato não suportado".',
        "A sincronização entre o celular e o computador parou de funcionar.",
        "O botão de salvar não responde no formulário de cadastro.",
        "Depois que atualizei o Android, o app não abre mais.",
        "O relatório mensal está mostrando dados duplicados.",
    ],
    ("tecnico", "negacao"): [
        "Não quero cancelar nada, só preciso que o app pare de fechar sozinho.",
        "Não é problema de senha: eu entro normalmente, mas a tela fica em branco depois.",
        "Minha fatura está correta; o que não funciona é a geração do relatório.",
        "O pedido não tem nada a ver com isso, o problema é o site que não carrega.",
        "A exportação não funciona nem no Chrome nem no Firefox.",
        "Não consigo anexar arquivos maiores que 2 MB, sempre dá erro.",
        "Não aparece nenhum dado no painel desde a atualização.",
        "Nenhuma das notificações push está chegando.",
        "Não fui cobrado errado; o app é que mostra o saldo zerado por erro.",
        "Não estou pedindo reembolso, quero que corrijam o erro na emissão do relatório.",
    ],
    ("tecnico", "ambiguo"): [
        "Tentei pagar, mas o botão de confirmar não funciona.",
        "O rastreio do pedido mostra erro na página.",
        "O app trava tanto que estou pensando em cancelar.",
        "Depois de trocar a senha, o app começou a travar na abertura.",
        "Não consigo emitir a segunda via porque a página dá erro.",
        "O site fica fora do ar e não consigo acompanhar a entrega.",
        "A tela de cadastro de novos usuários dá erro ao salvar.",
        "O sistema está lento e o suporte disse que é limitação do meu plano.",
    ],
    ("entrega", "direto"): [
        "Meu pedido está parado na transportadora há uma semana.",
        "O código de rastreio não mostra nenhuma atualização.",
        "O produto chegou quebrado.",
        "Recebi um item diferente do que comprei.",
        "A entrega estava prevista para ontem e não chegou.",
        "Quero trocar o endereço de entrega do pedido 4521.",
        "Faltou um item no pacote que recebi.",
        "O entregador marcou como entregue, mas não recebi nada.",
        "Quero trocar o tamanho da camiseta que chegou.",
        "Qual é o prazo de entrega para Manaus?",
        "A caixa chegou aberta e com sinais de violação.",
        "Preciso agendar a entrega para depois das 18h.",
    ],
    ("entrega", "negacao"): [
        "Não quero cancelar a compra, só quero saber quando o pedido vai chegar.",
        "Não preciso de reembolso, quero que enviem o item que faltou.",
        "O pagamento já foi aprovado; o problema é que o pedido não saiu do centro de distribuição.",
        "Ainda não recebi o pedido feito há dez dias.",
        "O pedido não chegou e o rastreio não atualiza.",
        "Não encontro a encomenda e o porteiro diz que não recebeu nada.",
        "Não é erro no app: o pedido realmente consta como devolvido ao remetente.",
        "Nenhum dos três volumes do pedido foi entregue.",
        "Não era esse o modelo que eu pedi.",
        "Não tenho problema com a conta, só quero mudar o endereço de entrega.",
    ],
    ("entrega", "ambiguo"): [
        "O produto chegou com defeito; quero trocar por outro igual.",
        "O pedido atrasou tanto que já não sei se ainda quero.",
        "Paguei o frete, mas o pedido ainda não foi enviado.",
        "Não consigo ver o status do pedido no app.",
        "O pedido chegou incompleto e o valor cobrado foi do pedido inteiro.",
        "Mudei de endereço e não sei se o pedido vai para o antigo.",
        "A transportadora diz que o endereço está errado, mas ele confere com meu cadastro.",
        "Recebi duas unidades e só comprei uma.",
    ],
    ("cancelamento", "direto"): [
        "Quero cancelar minha assinatura a partir do próximo mês.",
        "Por favor, encerrem minha conta e apaguem meus dados.",
        "Como faço para cancelar o plano anual?",
        "Desejo cancelar o pedido 7788 antes do envio.",
        "Quero desistir da compra dentro do prazo de sete dias.",
        "Cancelem a renovação automática do meu plano.",
        "Vou migrar para outro fornecedor e quero cancelar o contrato.",
        "Solicito o cancelamento da linha adicional.",
        "Quero cancelar o serviço de backup contratado no mês passado.",
        "Gostaria de encerrar a conta da empresa, que foi fechada.",
        "Peço o cancelamento imediato do contrato, conforme a cláusula de fidelidade.",
        "Quero cancelar a assinatura premium.",
    ],
    ("cancelamento", "negacao"): [
        "Não quero mais continuar com o plano; cancelem, por favor.",
        "Não tenho mais interesse no serviço e quero encerrar a assinatura.",
        "Não é problema técnico nem de cobrança: eu simplesmente quero cancelar.",
        "Não quero trocar o produto, quero cancelar o pedido.",
        "Não renovem meu plano; quero encerrar no fim do ciclo.",
        "Não preciso de desconto para ficar, quero cancelar mesmo.",
        "Não tenho tempo para usar o serviço, então prefiro cancelar.",
        "Não quero reembolso parcial, quero cancelar o contrato inteiro.",
        "Nunca mais vou usar, encerrem minha conta.",
        "Não quero suporte técnico, quero o cancelamento da assinatura.",
    ],
    ("cancelamento", "ambiguo"): [
        "Quero cancelar e receber de volta o valor do mês.",
        "Quero cancelar porque o app vive travando.",
        "Quero cancelar o pedido porque está atrasado demais.",
        "Quero reduzir meu plano para o básico ou cancelar, o que for mais barato.",
        "Encerrem minha conta, pois não consigo mais acessá-la mesmo.",
        "Quero cancelar a compra e devolver o produto que chegou errado.",
        "Desisti: cancelem tudo e estornem o que foi cobrado.",
        "Quero cancelar o plano empresarial e manter só o individual.",
    ],
    ("sem_informacao", "insuficiente"): [
        "Oi, preciso de ajuda.",
        "Não está funcionando.",
        "Bom dia, tenho um problema.",
        "Isso de novo?!",
        "Vocês podem me ligar?",
        "Já falei com vocês ontem sobre aquilo.",
        "Continua igual.",
        "Urgente!!!",
        "Não sei explicar, mas algo está errado.",
        "Quero falar com um atendente.",
        "O protocolo 3381 ainda não foi resolvido.",
        "Segue o print em anexo.",
        "???",
        "Por favor, verifiquem minha situação.",
        "Não deu certo.",
        "Oi",
        "Alguém pode me responder?",
        "Estou esperando há dias.",
        "Mesma coisa do chamado anterior.",
        "Tem como resolver isso hoje?",
        "Preciso de uma posição.",
        "Não entendi o que aconteceu.",
        "Teste",
        "Vocês resolveram?",
        "Quero fazer uma reclamação.",
        "Aconteceu de novo e não é a primeira vez.",
        "Olá, poderiam verificar?",
        "Ainda não.",
        "Estou muito insatisfeita.",
        "Me liga quando puder.",
    ],
}


def montar_dados(chamados, semente=2026, fracao_ajuste=0.4):
    import random
    linhas = []
    for (categoria, tipo), textos in chamados.items():
        for texto in textos:
            linhas.append({"id": f"CH{len(linhas) + 1:03d}", "texto": texto, "categoria": categoria, "tipo": tipo})
    rng, grupos = random.Random(semente), {}
    for i, linha in enumerate(linhas):
        grupos.setdefault((linha["categoria"], linha["tipo"]), []).append(i)
    for chave in sorted(grupos):
        indices = grupos[chave][:]
        rng.shuffle(indices)
        for j, i in enumerate(indices):
            linhas[i]["particao"] = "ajuste" if j < round(fracao_ajuste * len(indices)) else "avaliacao"
    return pd.DataFrame(linhas)


dados = montar_dados(CHAMADOS)
dados.to_csv(PASTA / "chamados_ptbr.csv", index=False)

In [ ]:
tipos = {"direto": AZUL, "negacao": LARANJA, "ambiguo": "#1baf7a", "insuficiente": "#eda100"}
tab = pd.crosstab(dados.categoria, dados.tipo).reindex(index=ROTULOS[::-1], columns=list(tipos), fill_value=0)

fig, ax = plt.subplots(figsize=(8, 3.6))
tab.plot.barh(stacked=True, ax=ax, color=list(tipos.values()), width=0.62, edgecolor=FUNDO, linewidth=2)
ax.grid(axis="y", visible=False)
ax.set(xlabel="chamados", ylabel="", title=f"{len(dados)} chamados · ajuste {(dados.particao == 'ajuste').sum()} · avaliação {(dados.particao == 'avaliacao').sum()}")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.2))
salvar(fig, "dados.png")
dados.groupby("tipo").head(1)

## Execução dos modelos

Rodo os dois modelos nos 180 chamados, um de cada vez, e registro versões, hardware, latência e pico de memória.

In [ ]:
def ambiente():
    import torch, transformers
    gpu = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
    return {"data_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "python": sys.version.split()[0],
            "torch": torch.__version__, "transformers": transformers.__version__, "sistema": platform.platform(),
            "gpu": gpu.name if gpu else None, "vram_gb": round(gpu.total_memory / 2**30, 1) if gpu else None}


def licenca(repo):
    try:
        from huggingface_hub import model_info
        info = model_info(repo)
        return {"repo": repo, "sha": info.sha, "licenca": (info.card_data or {}).get("license")}
    except Exception as erro:
        return {"repo": repo, "erro": str(erro)}


def cronometrar(funcao):
    import torch
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    inicio = time.perf_counter()
    saida = funcao()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return saida, (time.perf_counter() - inicio) * 1000


def memoria_mb():
    import torch, psutil
    if torch.cuda.is_available():
        return {"gpu_pico_mb": round(torch.cuda.max_memory_allocated() / 2**20)}
    return {"ram_processo_mb": round(psutil.Process().memory_info().rss / 2**20)}


def zerar_memoria():
    import gc, torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

In [ ]:
PERGUNTA = {"type": "choice", "instructions": INSTRUCAO, "criteria": CATEGORIAS}


def rodar_laya(dados):
    import laya
    from laya import Router
    zerar_memoria()
    router = Router(max_loaded=1)
    prever = lambda texto: router.predict(texto, {"categoria": PERGUNTA}, model=LAYA_CHECKPOINT)
    for texto in dados.texto.head(3):
        prever(texto)
    linhas = []
    for linha in dados.itertuples():
        resposta, ms = cronometrar(lambda: prever(linha.texto))
        r = resposta["answers"]["categoria"]
        linhas.append({"id": linha.id, "laya_pred": r["choice"], "laya_ms": ms, "laya_checkpoint": resposta["routing"]["model"],
                       **{f"laya_p_{c}": r["probabilities"][c] for c in ROTULOS}})
    agente = router.load(LAYA_CHECKPOINT)
    meta = {"versao": laya.__version__, "checkpoint": LAYA_CHECKPOINT, "device": str(agente.device),
            "precisao": str(getattr(agente, "dtype", "")), **memoria_mb(),
            **licenca(f"convaiinnovations/laya-{LAYA_CHECKPOINT}")}
    del router, agente
    return pd.DataFrame(linhas), meta

In [ ]:
SISTEMA = ("Você faz a triagem de chamados de atendimento ao cliente.\n"
           f"{INSTRUCAO} Escolha exatamente uma categoria:\n"
           + "\n".join(f"- {c}: {d}" for c, d in CATEGORIAS.items())
           + '\n\nResponda somente com JSON: {"categoria": "<chave>", "confianca": <número de 0 a 1>}.')


def sem_acento(texto):
    return "".join(c for c in unicodedata.normalize("NFD", str(texto).lower()) if unicodedata.category(c) != "Mn")


def ler_resposta(texto):
    import re
    for bloco in re.findall(r"\{[^{}]*\}", texto):
        try:
            obj = json.loads(bloco)
        except json.JSONDecodeError:
            continue
        cat = sem_acento(obj.get("categoria", "")).strip().replace(" ", "_")
        if cat in CATEGORIAS:
            try:
                conf = float(obj.get("confianca"))
                conf = conf / 100 if 1 < conf <= 100 else conf
            except (TypeError, ValueError):
                conf = np.nan
            return cat, conf if 0 <= conf <= 1 else np.nan, "json"
    return "invalido", np.nan, "invalido"


def rodar_gemma(dados):
    import torch
    from transformers import AutoModelForCausalLM
    try:
        from transformers import AutoProcessor
        processador = AutoProcessor.from_pretrained(GEMMA_MODELO)
    except Exception:
        from transformers import AutoTokenizer
        processador = AutoTokenizer.from_pretrained(GEMMA_MODELO)
    tok = getattr(processador, "tokenizer", processador)
    cuda = torch.cuda.is_available()
    dtype = torch.bfloat16 if not cuda or torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    zerar_memoria()
    modelo = AutoModelForCausalLM.from_pretrained(GEMMA_MODELO, dtype=dtype, device_map="auto" if cuda else None).eval()
    device = next(modelo.parameters()).device

    def prompt(texto):
        mensagens = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": f"Chamado: {texto}"}]
        return processador.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True, enable_thinking=False)

    def ids(texto):
        lista = tok(texto, add_special_tokens=False)["input_ids"]
        return lista if tok.bos_token_id is None or lista[:1] == [tok.bos_token_id] else [tok.bos_token_id] + lista

    @torch.inference_mode()
    def gerar(texto):
        entrada = torch.tensor([ids(prompt(texto))], device=device)
        saida = modelo.generate(input_ids=entrada, attention_mask=torch.ones_like(entrada), max_new_tokens=48, do_sample=False)
        return tok.decode(saida[0, entrada.shape[1]:], skip_special_tokens=True)

    @torch.inference_mode()
    def prob_rotulos(texto):
        prefixo = prompt(texto) + '{"categoria": "'
        base = ids(prefixo)
        logps = []
        for c in ROTULOS:
            completo = ids(prefixo + c + '"')
            comum = next((i for i, (a, b) in enumerate(zip(base, completo)) if a != b), len(base))
            n = len(completo) - comum
            alvo = torch.tensor([completo], device=device)
            logits = modelo(input_ids=alvo, logits_to_keep=n + 1).logits[0][-(n + 1):-1].float()
            logps.append(float(torch.log_softmax(logits, -1).gather(1, alvo[0, -n:, None]).sum()))
        e = np.exp(np.array(logps) - max(logps))
        return e / e.sum()

    assert np.isfinite(prob_rotulos("Fui cobrado duas vezes.")).all(), "Recebi probabilidades inválidas; preciso carregar o Gemma em torch.bfloat16 ou float32."
    for texto in dados.texto.head(3):
        gerar(texto)

    linhas = []
    for linha in dados.itertuples():
        bruta, ms = cronometrar(lambda: gerar(linha.texto))
        pred, conf, status = ler_resposta(bruta)
        p = prob_rotulos(linha.texto)
        linhas.append({"id": linha.id, "gemma_pred": pred, "gemma_conf_escrita": conf, "gemma_status": status,
                       "gemma_resposta": bruta, "gemma_ms": ms, **{f"gemma_p_{c}": v for c, v in zip(ROTULOS, p)}})
    meta = {"modelo": GEMMA_MODELO, "precisao": str(dtype), "device": str(device), "decodificacao": "gulosa, sem raciocínio",
            **memoria_mb(), **licenca(GEMMA_MODELO)}
    del modelo
    return pd.DataFrame(linhas), meta

In [ ]:
def resultados_salvos():
    try:
        return ler_csv("resultados/predicoes_laya.csv"), ler_csv("resultados/predicoes_gemma.csv"), ler_json("resultados/execucao.json")
    except Exception:
        return None


salvos = None if REFAZER else resultados_salvos()
if salvos:
    laya_df, gemma_df, execucao = salvos
    print("Li os resultados que já salvei.")
else:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Preciso de GPU: Ambiente de execução > Alterar o tipo de ambiente de execução > T4 GPU.")
    if "google.colab" in sys.modules:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", f"laya=={LAYA_VERSAO}", "transformers>=5", "accelerate"], check=True)
    print("Rodo o Laya...")
    laya_df, meta_laya = rodar_laya(dados)
    print("Rodo o Gemma...")
    gemma_df, meta_gemma = rodar_gemma(dados)
    laya_df.to_csv(PASTA / "predicoes_laya.csv", index=False)
    gemma_df.to_csv(PASTA / "predicoes_gemma.csv", index=False)
    execucao = {"ambiente": ambiente(), "laya": meta_laya, "gemma": meta_gemma}
    (PASTA / "execucao.json").write_text(json.dumps(execucao, indent=2, ensure_ascii=False), encoding="utf-8")

base = dados.merge(laya_df, on="id").merge(gemma_df, on="id")
aj, av = base[base.particao == "ajuste"], base[base.particao == "avaliacao"]
execucao

## Comparação na avaliação

Calculo intervalos de 95% por bootstrap e conto como erro toda saída inválida do Gemma.

In [ ]:
from scipy.stats import binomtest
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score

f1 = lambda y, p: f1_score(y, p, labels=ROTULOS, average="macro", zero_division=0)
acc = lambda y, p: float(np.mean(np.asarray(y) == np.asarray(p)))
memoria = lambda meta: meta.get("gpu_pico_mb", meta.get("ram_processo_mb"))


def intervalo(y, p, fn, n=2000):
    y, p, rng = np.asarray(y), np.asarray(p), np.random.default_rng(0)
    vals = [fn(y[i], p[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return np.percentile(vals, [2.5, 97.5])


y = av.categoria.to_numpy()
resumo = {}
for m in ["laya", "gemma"]:
    p = av[f"{m}_pred"].to_numpy()
    resumo[m] = {"acuracia": acc(y, p), "acuracia_ic": intervalo(y, p, acc), "f1": f1(y, p), "f1_ic": intervalo(y, p, f1),
                 "p50_ms": av[f"{m}_ms"].median(), "p95_ms": av[f"{m}_ms"].quantile(0.95)}

acerto_l, acerto_g = (av.laya_pred == av.categoria).to_numpy(), (av.gemma_pred == av.categoria).to_numpy()
so_l, so_g = int((acerto_l & ~acerto_g).sum()), int((~acerto_l & acerto_g).sum())
mcnemar = binomtest(so_l, so_l + so_g).pvalue if so_l + so_g else 1.0

pd.DataFrame({
    "acurácia": [f"{pct(r['acuracia'])} [{pct(r['acuracia_ic'][0])}; {pct(r['acuracia_ic'][1])}]" for r in resumo.values()],
    "F1 macro": [f"{pct(r['f1'])} [{pct(r['f1_ic'][0])}; {pct(r['f1_ic'][1])}]" for r in resumo.values()],
    "latência p50 (ms)": [round(r["p50_ms"], 1) for r in resumo.values()],
    "latência p95 (ms)": [round(r["p95_ms"], 1) for r in resumo.values()],
    "memória pico (MB)": [memoria(execucao["laya"]), memoria(execucao["gemma"])],
    "modelo": [f"laya-{execucao['laya']['checkpoint']}", execucao["gemma"]["modelo"]],
}, index=["Laya", "Gemma"])

In [ ]:
print(f"Em {so_l} chamados só o Laya acerta e em {so_g} só o Gemma acerta. No teste de McNemar exato, obtenho p = {mcnemar:.3f}.")
av.assign(Laya=acerto_l, Gemma=acerto_g).groupby("tipo")[["Laya", "Gemma"]].mean().style.format("{:.0%}")

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 5))
for ax, (nome, col) in zip(eixos, [("Laya", "laya_pred"), ("Gemma", "gemma_pred")]):
    colunas = ROTULOS + (["invalido"] if (av[col] == "invalido").any() else [])
    m = confusion_matrix(av.categoria, av[col], labels=colunas)[: len(ROTULOS)]
    ax.imshow(m, cmap="Blues")
    ax.grid(False)
    for (i, j), v in np.ndenumerate(m):
        if v:
            ax.text(j, i, v, ha="center", va="center", fontsize=9, color="white" if v > m.max() * 0.55 else TINTA)
    ax.set_xticks(range(len(colunas)), colunas, rotation=40, ha="right")
    ax.set_yticks(range(len(ROTULOS)), ROTULOS)
    ax.set(title=nome, xlabel="previsto", ylabel="referência")
fig.tight_layout()
salvar(fig, "matrizes_confusao.png")

In [ ]:
av.loc[~acerto_l | ~acerto_g, ["id", "texto", "categoria", "tipo", "laya_pred", "gemma_pred", "gemma_conf_escrita"]].sort_values("tipo")

## Confiabilidade dos escores

Ajusto uma temperatura no ajuste para o Laya e para a probabilidade que calculo dos logits do Gemma. A confiança que o Gemma escreve entra só como ordem (AUROC), porque não é uma probabilidade.

In [ ]:
from scipy.optimize import minimize_scalar

probs = lambda df, m: df[[f"{m}_p_{c}" for c in ROTULOS]].to_numpy(dtype=float)
indice = lambda serie: np.array([ROTULOS.index(c) for c in serie])


def com_temperatura(P, T):
    z = np.log(np.clip(P, 1e-6, 1)) / T
    e = np.exp(z - z.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def ajustar_temperatura(P, yi):
    nll = lambda lt: -np.mean(np.log(com_temperatura(P, np.exp(lt))[np.arange(len(yi)), yi]))
    return float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method="bounded").x))


def ece(conf, acerto, bins=10):
    grupos = np.clip((conf * bins).astype(int), 0, bins - 1)
    return sum(abs(acerto[grupos == b].mean() - conf[grupos == b].mean()) * (grupos == b).mean()
               for b in range(bins) if (grupos == b).any())


def auroc(conf, acerto):
    ok = ~np.isnan(conf)
    return roc_auc_score(acerto[ok], conf[ok]) if len(set(acerto[ok])) == 2 else np.nan


T = {m: ajustar_temperatura(probs(aj, m), indice(aj.categoria)) for m in ["laya", "gemma"]}
yi = indice(av.categoria)
sinais, linhas = {}, []
for m, nome in [("laya", "Laya"), ("gemma", "Gemma prob. calculada")]:
    for rotulo, P in [("bruto", probs(av, m)), (f"T={T[m]:.2f}", com_temperatura(probs(av, m), T[m]))]:
        conf, acerto = P.max(1), P.argmax(1) == yi
        sinais[f"{nome} {rotulo}"] = (conf, acerto)
        linhas.append({"sinal": f"{nome} {rotulo}", "ECE": ece(conf, acerto), "confiança média": conf.mean(),
                       "acurácia": acerto.mean(), "AUROC": auroc(conf, acerto)})
escrita = av.gemma_conf_escrita.to_numpy(dtype=float)
linhas.append({"sinal": "Gemma confiança escrita", "AUROC": auroc(escrita, acerto_g),
               "valores distintos": len(np.unique(escrita[~np.isnan(escrita)]))})
tabela_escores = pd.DataFrame(linhas).round(3)
tabela_escores

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5.6))
ax.plot([0, 1], [0, 1], ":", color=TINTA_SUAVE, label="calibração perfeita")
for nome, cor, estilo in [("Laya bruto", AZUL, ":"), (f"Laya T={T['laya']:.2f}", AZUL, "-"),
                          (f"Gemma prob. calculada T={T['gemma']:.2f}", LARANJA, "-")]:
    conf, acerto = sinais[nome]
    pontos = (pd.DataFrame({"g": np.clip((conf * 10).astype(int), 0, 9), "c": conf, "a": acerto})
              .groupby("g").filter(lambda g: len(g) >= 3).groupby("g")[["c", "a"]].mean())
    ax.plot(pontos.c, pontos.a, estilo, marker="o", color=cor, lw=2, ms=7, mec=FUNDO, mew=2,
            label=f"{nome} (ECE {ece(conf, acerto):.3f})")
ax.set(xlim=(0, 1.02), ylim=(0, 1.02), xlabel="probabilidade do rótulo escolhido", ylabel="acurácia observada",
       title="Confiabilidade na avaliação")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2)
salvar(fig, "confiabilidade.png")

## Política de revisão humana

Mando `sem_informacao` sempre para uma pessoa. Automatizo o resto quando a probabilidade calibrada do Laya chega ao limiar τ. Escolho o menor τ que mantém o erro em até `ERRO_ALVO` no ajuste.

In [ ]:
def politica(conf, pred, limiar):
    return ~np.isin(pred, REVISAR_SEMPRE) & (conf >= limiar)


conf_aj = com_temperatura(probs(aj, "laya"), T["laya"]).max(1)
erro_aj = (aj.laya_pred != aj.categoria).to_numpy()
TAU = 1.01
for t in np.sort(np.unique(conf_aj)):
    auto = politica(conf_aj, aj.laya_pred, t)
    if auto.sum() < 10:
        break
    if erro_aj[auto].mean() <= ERRO_ALVO:
        TAU = float(t)
        break

conf_av = com_temperatura(probs(av, "laya"), T["laya"]).max(1)
decisao = av[["id", "texto", "categoria", "tipo", "laya_pred"]].assign(
    probabilidade=conf_av.round(3), acao=np.where(politica(conf_av, av.laya_pred, TAU), "automatico", "revisao"),
    acertou=acerto_l)
auto = (decisao.acao == "automatico").to_numpy()
res_politica = {"limiar": TAU, "n": len(decisao), "automatizados": int(auto.sum()),
                "erros_automatizados": int((auto & ~acerto_l).sum()), "revisao": int((~auto).sum()),
                "erros_capturados": int((~auto & ~acerto_l).sum()), "erros_totais": int((~acerto_l).sum())}
display(Markdown(
    f"Com τ = **{TAU:.2f}**, automatizo **{res_politica['automatizados']} de {res_politica['n']}** chamados "
    f"({pct(auto.mean())}) e restam **{res_politica['erros_automatizados']} erros** entre eles. "
    f"A revisão recebe {res_politica['revisao']} chamados e captura {res_politica['erros_capturados']} dos {res_politica['erros_totais']} erros do Laya."))
pd.crosstab(decisao.tipo, [decisao.acao, decisao.acertou])

In [ ]:
decisao[auto & ~acerto_l]

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 5))
gemma_cal = sinais[f"Gemma prob. calculada T={T['gemma']:.2f}"]
curvas = [("Laya, prob. calibrada", conf_av, acerto_l, AZUL, "-"),
          ("Gemma, prob. calculada", gemma_cal[0], gemma_cal[1], LARANJA, "-"),
          ("Gemma, confiança escrita", np.nan_to_num(escrita, nan=-1), acerto_g, LARANJA, "--")]
for nome, conf, acerto, cor, estilo in curvas:
    ordem = np.argsort(-conf, kind="stable")
    erro = np.cumsum(~acerto[ordem]) / np.arange(1, len(ordem) + 1)
    ax.step(np.arange(1, len(ordem) + 1) / len(ordem), erro, estilo, where="post", color=cor, lw=2, label=nome)
ax.axhline(ERRO_ALVO, ls="--", lw=1, color=TINTA_SUAVE)
ax.set(xlim=(0, 1), ylim=(0, None), xlabel="fração automatizada (da maior para a menor confiança)",
       ylabel="erro entre os automatizados", title="Risco × cobertura na avaliação")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3)
salvar(fig, "risco_cobertura.png")

## Destaque e exportação

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(13, 3.9), gridspec_kw={"width_ratios": [1.25, 0.9, 1.35]})
ax = eixos[0]
for j, (m, nome, cor) in enumerate([("laya", "Laya", AZUL), ("gemma", "Gemma", LARANJA)]):
    r, xs = resumo[m], np.arange(2) + (j - 0.5) * 0.36
    vals, ics = [r["acuracia"], r["f1"]], [r["acuracia_ic"], r["f1_ic"]]
    ax.bar(xs, vals, 0.34, color=cor, label=nome, edgecolor=FUNDO, lw=2)
    ax.errorbar(xs, vals, yerr=np.array([[v - i[0], i[1] - v] for v, i in zip(vals, ics)]).T, fmt="none", ecolor=TINTA_SUAVE, capsize=3)
    for x, v in zip(xs, vals):
        ax.text(x, 0.03, f"{v:.0%}", ha="center", color="white", fontweight="bold")
ax.set(xticks=[0, 1], ylim=(0, 1.18), yticks=np.linspace(0, 1, 6), title="Qualidade na avaliação (IC 95%)")
ax.set_xticklabels(["acurácia", "F1 macro"])
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.grid(axis="x", visible=False)
ax.legend(loc="upper left", ncol=2)

ax = eixos[1]
lat = [resumo["gemma"]["p50_ms"], resumo["laya"]["p50_ms"]]
ax.barh(["Gemma", "Laya"], lat, color=[LARANJA, AZUL], height=0.55)
for i, v in enumerate(lat):
    ax.text(v, i, f"  {v:,.0f} ms".replace(",", "."), va="center", color=TINTA)
ax.set(xlim=(0, max(lat) * 1.45), title="Latência p50 por chamado")
ax.grid(axis="y", visible=False)

ax = eixos[2]
partes = [("automatizado e certo", res_politica["automatizados"] - res_politica["erros_automatizados"], AZUL),
          ("automatizado e errado", res_politica["erros_automatizados"], VERMELHO), ("revisão humana", res_politica["revisao"], CINZA)]
inicio = 0
for nome, v, cor in partes:
    ax.barh([0], [v], left=inicio, color=cor, edgecolor=FUNDO, lw=2, height=0.5, label=f"{nome}: {v}")
    if v >= 0.06 * res_politica["n"]:
        ax.text(inicio + v / 2, 0, v, ha="center", va="center", fontweight="bold", color=TINTA if cor == CINZA else "white")
    inicio += v
ax.set(xlim=(0, res_politica["n"]), ylim=(-0.9, 0.9), yticks=[], xlabel=f"{res_politica['n']} chamados de avaliação",
       title=f"Laya + revisão humana (erro-alvo {ERRO_ALVO:.0%})")
ax.grid(False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.32), ncol=2)
fig.tight_layout()
salvar(fig, "destaque.png")

In [ ]:
politica_laya = {"checkpoint": LAYA_CHECKPOINT, "laya_versao": execucao["laya"]["versao"], "instrucao": INSTRUCAO,
                 "categorias": CATEGORIAS, "temperatura": T["laya"], "limiar": TAU, "erro_alvo": ERRO_ALVO,
                 "revisar_sempre": REVISAR_SEMPRE}
(PASTA / "politica_laya.json").write_text(json.dumps(politica_laya, indent=2, ensure_ascii=False), encoding="utf-8")

L, G = resumo["laya"], resumo["gemma"]
texto = f'''| | Laya | Gemma |
|---|---|---|
| modelo | `laya-{LAYA_CHECKPOINT}` {execucao["laya"]["versao"]} | `{execucao["gemma"]["modelo"]}` |
| acurácia | {pct(L["acuracia"])} [{pct(L["acuracia_ic"][0])}; {pct(L["acuracia_ic"][1])}] | {pct(G["acuracia"])} [{pct(G["acuracia_ic"][0])}; {pct(G["acuracia_ic"][1])}] |
| F1 macro | {pct(L["f1"])} | {pct(G["f1"])} |
| latência p50 | {L["p50_ms"]:.0f} ms | {G["p50_ms"]:.0f} ms |
| memória pico | {memoria(execucao["laya"])} MB | {memoria(execucao["gemma"])} MB |

Com τ = {TAU:.2f}, automatizo {res_politica["automatizados"]} de {res_politica["n"]} chamados e restam {res_politica["erros_automatizados"]} erros entre eles. Rodei os modelos em {execucao["ambiente"]["gpu"] or "CPU"} em {execucao["ambiente"]["data_utc"][:10]}.
'''
(PASTA / "resumo.md").write_text(texto, encoding="utf-8")
display(Markdown(texto))

if not salvos and "google.colab" in sys.modules:
    import shutil
    from google.colab import files
    files.download(shutil.make_archive("resultados", "zip", PASTA))